Copie de ODL_lab_transformers_2026.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1dliYWsrIThhMA78u_HR6ZC8f-_8qYKsS

# TP — Transformers : mini-GPT from scratch (PyTorch)


**Partie A (petit transformer, from scratch avec briques PyTorch)**
- On implémente un petit GPT causal (embeddings, self-attention, multi-head, blocs, génération).
- On entraîne sur un petit corpus texte (Tiny Shakespeare en char-level).
- On fait des ablations simples (profondeur, têtes, contexte, dropout).

**Partie B (appli “réaliste”, GPU recommandé)**
- On fine-tune un modèle **pré-entraîné** (GPT-2) avec Hugging Face `transformers`.
- Objectif : comparer “entraîner de zéro” vs “adapter un modèle pré-entraîné”.
- On observe la différence de qualité de génération avant/après fine-tuning (même si le fine-tuning est court).


---

In [ ]:
# Setup (Partie A)
import math
import os
import time
import random
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(0)
random.seed(0)

## Rappels : tokenizer, vocabulaire, embeddings

### Tokenizer : transformer du texte en entiers
Un **tokenizer** convertit un texte (chaîne de caractères) en une suite de **tokens**, puis en une suite d’**entiers**.

Pourquoi ? Parce qu’un réseau de neurones manipule des nombres, pas des caractères ou des mots bruts.

Dans ce TP, on utilise une tokenisation **char-level** (par caractère) :

- Texte : `"hello"`
- Tokens : `["h","e","l","l","o"]`
- IDs : `[17, 4, 11, 11, 14]` (exemple)

On construit :
- `stoi` (*string to int*) : caractère -> id
- `itos` (*int to string*) : id -> caractère
- `vocab_size` : nombre total de caractères distincts

Cette tokenisation est volontairement simple (pas de BPE), pour se concentrer sur le Transformer.

---

### Embedding : convertir un id en vecteur
Un **embedding** est une table apprise qui transforme un entier (id de token) en un vecteur continu.

- Entrée : un entier `id` (0 à `vocab_size-1`)
- Sortie : un vecteur de dimension `n_embd`

En PyTorch :
- `nn.Embedding(vocab_size, n_embd)` est une matrice de taille `(vocab_size, n_embd)`
- quand on donne des ids, on récupère les lignes correspondantes.

Intuition : on apprend une représentation vectorielle pour chaque token.

---

### Positional embedding : donner l’information d’ordre
La self-attention ne connaît pas l’ordre des tokens “naturellement”.
On ajoute donc une information de position.

Ici : embeddings de position **appris** :
- `pos_emb = nn.Embedding(block_size, n_embd)`
- on additionne : `tok_emb(idx) + pos_emb(pos)`

---

### Langage modèle causal : quel est le problème appris ?
Un GPT est un **modèle causal** : il prédit le prochain token.

Si on a une séquence :
$$
x_0, x_1, \dots, x_{T-1}
$$
on entraîne le modèle à prédire :
$$
x_1, x_2, \dots, x_T
$$

C’est pourquoi, dans le TP :
- `x` est un bloc de longueur `block_size`
- `y` est le même bloc décalé d’un token (“shift”)

La loss est une **cross-entropy** sur toutes les positions (toutes les prédictions de prochain token).

---

### Masque causal : empêcher de “voir le futur”
Sans masque, le modèle pourrait utiliser les tokens futurs (triche).
Le **masque causal** impose qu’à la position `t`, on ne peut regarder que les positions `0..t`.

C’est l’ingrédient clé qui fait d’un Transformer un GPT (côté entraînement comme côté génération).

## A2 — Données (Tiny Shakespeare) + tokenisation char-level

Le code ci-dessous essaie de télécharger Tiny Shakespeare. En cas d’échec (pas d’internet), il utilise un petit texte de fallback.

In [ ]:
def load_text():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    local_path = "tinyshakespeare.txt"
    if not os.path.exists(local_path):
        try:
            print("Downloading tinyshakespeare...")
            urllib.request.urlretrieve(url, local_path)
        except Exception as e:
            print("Download failed, using fallback text. Reason:", repr(e))
            return (
                "To be, or not to be: that is the question.\n"
                "Whether 'tis nobler in the mind to suffer\n"
                "The slings and arrows of outrageous fortune...\n"
            )
    with open(local_path, "r", encoding="utf-8") as f:
        return f.read()

text = load_text()
print("Text length:", len(text))
print(text[:400])

# Vocab char-level
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s: str):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join([itos[i] for i in ids])

data = torch.tensor(encode(text), dtype=torch.long)
print("vocab_size:", vocab_size, "| data shape:", data.shape)

# NB: the next cell REPLACES the char-level tokenisation defined above with a
# Byte-Level BPE. Both are kept in the notebook so they can be compared:
#   - char-level: tiny vocabulary (~65), very long sequences, the model has to
#     learn the spelling of words;
#   - BPE       : 5000-subword vocabulary, sequences ~4x shorter for the same
#     text, so a block_size of 256 covers far more context.
# The trade-off is that the embedding table grows from 65 to 5000 rows.
import torch
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# 1. Initialiser un BPE vide (pas besoin de [UNK] en Byte-Level)
bpe_tokenizer = Tokenizer(BPE())

# 2. Utiliser ByteLevel pour le pre-tokenizing ET le decoding
# C'est ce qui va préserver tes sauts de ligne (\n) et tes espaces
bpe_tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = ByteLevelDecoder()

# 3. Configurer l'entraîneur avec ta limite de 5000 tokens
# On utilise un token spécial de fin de texte standard pour les LLMs
trainer = BpeTrainer(
    vocab_size=5000,
    special_tokens=["<|endoftext|>", "[BOS]"] # Added [BOS] for compatibility, but <|endoftext|> is preferred
)

# 4. Entraîner sur tes données
print("Entraînement du Byte-Level BPE en cours...")
bpe_tokenizer.train_from_iterator([text], trainer=trainer)

# 5. Redéfinir tes fonctions
def encode(s: str):
    return bpe_tokenizer.encode(s).ids

def decode(ids):
    return bpe_tokenizer.decode(ids)

# 6. Créer ton tenseur de données
data = torch.tensor(encode(text), dtype=torch.long)
vocab_size = bpe_tokenizer.get_vocab_size()

print("vocab_size:", vocab_size, "| data shape:", data.shape)

## A3 — Batch sampling (LM causal)

On entraîne sur des blocs de longueur `block_size`.
- `x` : tokens [t, t+1, ..., t+block_size-1]
- `y` : tokens [t+1, ..., t+block_size] (décalage d’un token)

Le modèle apprend donc à prédire `y` à partir de `x`.

In [ ]:
# Hyperparamètres (CPU-friendly)
config = dict(
    block_size=256,
    batch_size=32,
    n_embd=128,
    n_heads=4,
    n_layers=8,
    dropout=0.1,
    lr=3e-4,
    max_iters=300,
    eval_interval=50,
    eval_iters=50,
)

# Split train/val
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - config["block_size"] - 1, (config["batch_size"],))
    x = torch.stack([source[i:i+config["block_size"]] for i in ix])
    y = torch.stack([source[i+1:i+config["block_size"]+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("xb:", xb.shape, "yb:", yb.shape)

## A4 — Self-attention causale : version naïve

Avant d’écrire une version “rapide” (vectorisée), on présente une version **naïve** :
- on calcule explicitement, pour chaque position `t`, une distribution d’attention sur les positions `0..t`,
- puis on fait une somme pondérée des `v`.

Cette version est **trop lente** pour de vrais modèles, mais elle est utile pour comprendre.

Notes :
- Ici on fait **single-head** pour être le plus clair possible.
- Plus bas, on passera à **multi-head** et à une version vectorisée.

In [ ]:
class NaiveCausalSelfAttention(nn.Module):
    """Self-attention causale naïve (single-head), lente.

    Objectif :
    Implémenter une attention causale "à la main", sans utiliser de masque
    triangulaire global ni de produit matriciel complet sur toute la séquence.

    L'idée est d'unroller la séquence temporelle :
        for t in range(T)

    À chaque instant t, le token x[:, t, :] ne doit regarder que :
        x[:, 0, :], x[:, 1, :], ..., x[:, t, :]

    Il ne doit jamais regarder les tokens futurs :
        x[:, t+1, :], ..., x[:, T-1, :]
    """

    def __init__(self, n_embd, block_size, dropout):
        super().__init__()
        self.n_embd = n_embd
        self.block_size = block_size
        self.dropout = dropout
        self.Q=nn.Linear(n_embd, n_embd)
        self.K=nn.Linear(n_embd, n_embd)
        self.V=nn.Linear(n_embd, n_embd)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        assert C == self.n_embd
        assert T <= self.block_size

        # -------------------------------------------------
        # Étape 1 : calculer les projections Q, K et V
        # -------------------------------------------------
        # À partir de x, calculer :
        #   Q, K, V
        # Utiliser ici les couches définies dans __init__.
        Q = self.Q(x)   # (B, T, C)
        K = self.K(x)  # (B, T, C)
        V = self.V(x)  # (B, T, C)

        # Tensor qui contiendra la sortie de l'attention
        Y = torch.zeros_like(x)  # (B, T, C)

        # -------------------------------------------------
        # Étape 2 : unroll temporel de la séquence
        # -------------------------------------------------
        for t in range(T):
            # query du token courant
            # (B, C)
            q_t = Q[:, t, :]

            # keys jusqu'au temps t inclus
            # (B, t+1, C)
            k_hist = K[:, :t+1, :]

            # scores d'attention
            # (B, t+1)
            # Ne pas oublier de normaliser sqrt(C).
            scores = (q_t.unsqueeze(1) @ k_hist.transpose(1, 2)).squeeze(1) / math.sqrt(C)

            # transformer les scores en poids d'attention: softmax sur la dimension temporelle.
            #     (B, t+1)
            att = F.softmax(scores, dim=-1)

            # dropout
            att = self.attn_dropout(att)

            # récupérer les values jusqu'au temps t inclus
            # (B, t+1, C)
            v_hist = V[:, :t+1, :]

            # calculer la sortie au temps t
            # (B, C)
            y_t = (att.unsqueeze(1) @ v_hist).squeeze(1)

            # Stocker y_t dans la sortie Y à la position t.
            Y[:, t, :] = y_t

        # -------------------------------------------------
        # Étape 3 : projection finale + dropout résiduel
        # -------------------------------------------------
        Y = self.resid_dropout(self.proj(Y))

        return Y

x=torch.randn([10,5,10])
naive_attention=NaiveCausalSelfAttention(10,5,0.1)
y=naive_attention(x)
print(x.shape)
print(y.shape)

## A5 — Self-attention causale : version vectorisée multi-head

Maintenant on implémente la version standard (style GPT).

### TODO 1
Compléter `CausalSelfAttention.forward`.

In [ ]:
class CausalSelfAttention(nn.Module):
    """Self-attention causale multi-head.

    Objectif :
    Implémenter l'attention causale de manière vectorisée, avec plusieurs têtes.

    Contrairement à la version naïve, on ne fait pas de boucle sur le temps.
    On calcule directement tous les scores d'attention entre toutes les positions,
    puis on applique un masque causal pour interdire à un token de regarder le futur.
    """

    def __init__(self, n_embd, n_heads, block_size, dropout):
        super().__init__()
        assert n_embd % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = n_embd // n_heads

        # A SINGLE projection producing q, k and v at once: this is
        # mathematically equivalent to three separate nn.Linear layers (their
        # matrices are simply concatenated), but it is one matrix product
        # instead of three -> markedly faster on GPU.
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        # masque causal de taille (block_size, block_size)
        # il contient des 1 sur le triangle inférieur, et des 0 ailleurs
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("causal_mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        """x: (B,T,C) -> (B,T,C)"""
        B, T, C = x.shape

        # -------------------------------------------------
        # Étape 1 : projeter x en Q, K, V
        # -------------------------------------------------
        # Utiliser la couche self.qkv pour obtenir un tenseur de taille (B, T, 3C),
        # puis le séparer en q, k et v.
        qkv = self.qkv(x)            # (B, T, 3C)
        q, k, v = qkv.chunk(3,dim=-1)         # chacun : (B, T, C)

        # -------------------------------------------------
        # Étape 2 : découper en plusieurs têtes
        # -------------------------------------------------
        # Réorganiser q, k et v pour obtenir :
        #     (B, n_heads, T, head_dim)
        q = q.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.head_dim).transpose(1,2)

In [ ]:
        # -------------------------------------------------
        # Étape 3 : calculer les scores d'attention
        # -------------------------------------------------
        # Calculer les produits scalaires entre queries et keys.
        # Résultat attendu :
        #     (B, n_heads, T, T)
        # Ne pas oublier la normalisation par sqrt(head_dim).
        scores = (q @ k.transpose(-2,-1))/math.sqrt(self.head_dim)

        # -------------------------------------------------
        # Étape 4 : appliquer le masque causal
        # -------------------------------------------------
        # Les positions futures doivent recevoir -inf avant le softmax.
        scores = scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        # -------------------------------------------------
        # Étape 5 : softmax + dropout
        # -------------------------------------------------
        # Le softmax se fait sur la dernière dimension.
        att = F.softmax(scores, dim=-1)
        att = self.attn_drop(att)

In [ ]:
        # -------------------------------------------------
        # Étape 6 : appliquer l'attention aux values
        # -------------------------------------------------
        # Résultat attendu :
        #     (B, n_heads, T, head_dim)
        y = att @ v

        # -------------------------------------------------
        # Étape 7 : recoller les têtes
        # -------------------------------------------------
        # Revenir à une représentation de taille :
        #     (B, T, C)
        y = y.transpose(1,2).contiguous().view(B,T,C)

        # -------------------------------------------------
        # Étape 8 : projection finale + dropout résiduel
        # -------------------------------------------------
        y = self.resid_drop(self.proj(y))

        return y

## A6 — Bloc Transformer + modèle GPT

### TODO

In [ ]:
# Dans cette partie, on assemble les briques précédentes pour construire un mini-GPT.
# Le MLP agit indépendamment sur chaque position, le TransformerBlock combine attention
# causale et MLP avec connexions résiduelles, et MiniGPT empile plusieurs blocs pour
# prédire le prochain token à chaque position.

class MLP(nn.Module):
    """MLP utilisé dans un bloc Transformer.

    Objectif :
    Implémenter le petit réseau feed-forward appliqué indépendamment
    à chaque position de la séquence.

    Il augmente d'abord la dimension cachée, applique une non-linéarité,
    puis revient à la dimension initiale.
    """

    def __init__(self, n_embd, dropout):
        super().__init__()

        # Todo
        # - première couche linéaire : n_embd -> 4 * n_embd
        # - deuxième couche linéaire : 4 * n_embd -> n_embd
        # - dropout final
        self.premierecouche=nn.Linear(n_embd,4*n_embd)
        self.deuxiemecouche=nn.Linear(4*n_embd,n_embd)
        self.lastdropout=nn.Dropout(dropout)

    def forward(self, x):
        # première projection linéaire
        x = self.premierecouche(x)

        # non-linéarité GELU
        x = F.gelu(x)

        # deuxième projection linéaire
        x = self.deuxiemecouche(x)

        # dropout
        x = self.lastdropout(x)

        return x

In [ ]:
class TransformerBlock(nn.Module):
    """Bloc Transformer pré-norm.

    Objectif :
    Implémenter un bloc Transformer composé de :
    - une self-attention causale
    - un MLP
    - deux LayerNorm
    - deux connexions résiduelles

    La normalisation est appliquée avant chaque sous-module :
        x + module(LayerNorm(x))
    """

    def __init__(self, n_embd, n_heads, block_size, dropout):
        super().__init__()

        # Todo
        # - LayerNorm avant l'attention
        # - attention causale multi-head
        # - LayerNorm avant le MLP
        # - MLP
        self.ln1=nn.LayerNorm(n_embd)
        self.ln2=nn.LayerNorm(n_embd)
        self.attention=CausalSelfAttention(n_embd,n_heads,block_size,dropout)
        self.mlp=MLP(n_embd,dropout)
    def forward(self, x):
        # bloc attention avec connexion résiduelle
        x = x+self.attention(self.ln1(x))

        # bloc MLP avec connexion résiduelle
        x = x+self.mlp(self.ln2(x))

        return x

In [ ]:
class MiniGPT(nn.Module):
    """Mini GPT pour la modélisation autoregressive de séquences.

    Objectif :
    Implémenter un petit modèle de langage de type GPT.

    Le modèle prend une séquence d'indices de tokens :
        idx : (B, T)

    et renvoie des logits de prédiction :
        logits : (B, T, vocab_size)

    Si des targets sont fournies, il calcule aussi la loss de cross-entropy.
    """

    def __init__(self, vocab_size, config):
        super().__init__()

        # Todo
        # - embedding des tokens
        # - embedding des positions
        # - dropout initial
        # - liste de blocs Transformer
        # - LayerNorm finale
        # - tête de prédiction vers le vocabulaire
        self.vocab_size = vocab_size
        self.block_size = config["block_size"]
        self.n_embd = config["n_embd"]
        self.n_heads = config["n_heads"]
        self.n_layers = config["n_layers"]
        self.dropout = config["dropout"]

In [ ]:
        self.token_embeddings = nn.Embedding(vocab_size, self.n_embd)
        self.position_embeddings = nn.Embedding(self.block_size, self.n_embd)
        self.drop = nn.Dropout(self.dropout)
        self.blocks = nn.Sequential(*[TransformerBlock(self.n_embd, self.n_heads, self.block_size, self.dropout) for _ in range(self.n_layers)])
        self.ln_final = nn.LayerNorm(self.n_embd)
        self.head = nn.Linear(self.n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size

        # -------------------------------------------------
        # Étape 1 : embeddings de tokens et de positions
        # -------------------------------------------------
        # idx contient des indices de tokens de taille (B, T).
        # Créer aussi les positions 0, ..., T-1 sur le bon device.
        pos = torch.arange(0,T,device=idx.device)

        # Combiner les embeddings de tokens et de positions.
        # Résultat attendu :
        #     x : (B, T, n_embd)
        token_emb = self.token_embeddings(idx)
        pos_emb = self.position_embeddings(pos)
        x = token_emb + pos_emb

        # dropout initial
        x = self.drop(x)

        # -------------------------------------------------
        # Étape 2 : passer dans les blocs Transformer
        # -------------------------------------------------
        for block in self.blocks:
            x = block(x)

        # -------------------------------------------------
        # Étape 3 : normalisation finale et prédiction
        # -------------------------------------------------
        x = self.ln_final(x)

        # logits : (B, T, vocab_size)
        logits = self.head(x)

        # -------------------------------------------------
        # Étape 4 : calcul optionnel de la loss
        # -------------------------------------------------
        loss = None

        if targets is not None:
            # Reformer logits et targets pour utiliser F.cross_entropy.
            #
            # logits doit devenir :
            #     (B*T, vocab_size)
            #
            # targets doit devenir :
            #     (B*T,)
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))

        return logits, loss

## A7 — Tests rapides

Ces tests doivent passer avant d’entraîner.

In [ ]:
model = MiniGPT(vocab_size, config).to(device)
xb, yb = get_batch("train")
logits, loss = model(xb, yb)
print("logits:", logits.shape)
print("loss:", loss.item() if loss is not None else None)
print(math.exp(loss.item()))

## A8 — Génération autoregressive

### TODO 3
Compléter `generate`.

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens=200, temperature=1.0, greedy=False):
    """Génération autoregressive de tokens.

    Objectif :
    Générer de nouveaux tokens un par un à partir d'un contexte initial idx.

    À chaque étape :
    - on garde seulement les derniers tokens compatibles avec block_size
    - on passe ce contexte dans le modèle
    - on récupère les logits du dernier pas de temps
    - on transforme ces logits en distribution de probabilité
    - on choisit le prochain token
    - on l'ajoute à la séquence
    """

    model.eval()

    for _ in range(max_new_tokens):

        idx_cond = idx if idx.size(1) <= model.block_size else idx[:, -model.block_size:]

        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        if temperature == 0.0 or greedy:
            probs = F.softmax(logits, dim=-1)
            next_token = torch.argmax(probs, dim=-1, keepdim=True)
        else:
            logits = logits / temperature

            probs = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_token), dim=1)

    return idx

## A9 — Entraînement (mini)

Lancez l’entraînement après avoir complété les TODOs.

In [ ]:
config = dict(
    block_size=256,
    batch_size=32,
    n_embd=128,
    n_heads=4,
    n_layers=8,
    dropout=0.1,
    lr=3e-4,
    max_iters=300,
    eval_interval=50,
    eval_iters=50,
)

In [ ]:
model = MiniGPT(vocab_size, config).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = []
        for _ in range(config["eval_iters"]):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

train_losses = []
val_losses = []
iters = []

t0 = time.time()
model.train()
for it in range(1, config["max_iters"] + 1):
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if it % config["eval_interval"] == 0 or it == 1:
        losses = estimate_loss()
        print(f"iter {it:04d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
        iters.append(it)
        train_losses.append(losses["train"])
        val_losses.append(losses["val"])

print("Training time (s):", round(time.time() - t0, 1))

plt.figure(figsize=(6,4))
plt.plot(iters, train_losses, label="train")
plt.plot(iters, val_losses, label="val")
plt.xlabel("iteration")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()
plt.show()

xb, yb = get_batch("train")
logits, loss = model(xb, yb)
print("logits:", logits.shape)
print("loss:", loss.item() if loss is not None else None)
print(math.exp(loss.item()))

## A10 — Génération (qualitative)

In [ ]:
prompt = "\n"
encoded_prompt = encode(prompt)

# If the encoded prompt is empty, add the [BOS] token
if not encoded_prompt:
    # Use the actual special token defined in the bpe_tokenizer
    bos_token_id = bpe_tokenizer.token_to_id("<|endoftext|>") # Changed to use bpe_tokenizer and its special token
    if bos_token_id is None:
        # Fallback if <|endoftext|> is not in vocab, or handle differently
        raise ValueError("Special token <|endoftext|> not found in tokenizer vocabulary. Cannot start generation with empty prompt.")
    idx0 = torch.tensor([[bos_token_id]], dtype=torch.long, device=device)
else:
    idx0 = torch.tensor([encoded_prompt], dtype=torch.long, device=device)

out = generate(model, idx0, max_new_tokens=300, temperature=1.0, greedy=False)
print(decode(out[0].tolist()))

## A11 — Ablations (2–3 expériences)

Faites 2–3 modifications dans `config` et relancez l'entraînement (plus court si CPU).

Automating this: a function that trains a model from a config and returns the
final validation loss. **Only one parameter at a time** is changed relative to
the reference config.

In [ ]:
def entrainer(cfg, max_iters=200, seed=0, verbose=False):
    """Train a MiniGPT and return (final loss, time, number of parameters)."""
    torch.manual_seed(seed)
    m = MiniGPT(vocab_size, cfg).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=cfg["lr"])
    t0 = time.time()
    m.train()
    for it in range(max_iters):
        # get_batch reads the global config variable, so keep it in sync
        globals()["config"] = cfg
        xb, yb = get_batch("train")
        _, loss = m(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    m.eval()
    pertes = []
    with torch.no_grad():
        for _ in range(20):
            xb, yb = get_batch("val")
            _, l = m(xb, yb)
            pertes.append(l.item())
    val = sum(pertes) / len(pertes)
    return dict(val_loss=val, perplexite=math.exp(val), temps=time.time() - t0,
                params=sum(p.numel() for p in m.parameters()), model=m)

In [ ]:
base = dict(block_size=128, batch_size=16, n_embd=128, n_heads=4, n_layers=4,
            dropout=0.1, lr=3e-4, max_iters=200, eval_interval=50, eval_iters=20)

ablations = {
    "reference":              dict(),
    "profondeur : 2 blocs":   dict(n_layers=2),
    "profondeur : 8 blocs":   dict(n_layers=8),
    "tetes : 1":              dict(n_heads=1),
    "tetes : 8":              dict(n_heads=8),
    "contexte : 32":          dict(block_size=32),
    "contexte : 256":         dict(block_size=256),
    "dropout : 0.0":          dict(dropout=0.0),
    "dropout : 0.3":          dict(dropout=0.3),
    "largeur : 64":           dict(n_embd=64),
}

resultats = {}
for nom, modif in ablations.items():
    cfg = dict(base); cfg.update(modif)
    r = entrainer(cfg, max_iters=200)
    resultats[nom] = r
    print(f"{nom:24s} | {r['params']:>9,d} params | val loss {r['val_loss']:.4f}"
          f" | perplexite {r['perplexite']:6.1f} | {r['temps']:5.1f} s")

globals()["config"] = base   # restore the reference config

plt.figure(figsize=(10, 4))
noms = list(resultats.keys())
plt.barh(noms, [resultats[n]['val_loss'] for n in noms])
plt.xlabel("validation loss (lower is better)")
plt.title("Mini-GPT ablations (200 iterations each)")
plt.grid(alpha=.3, axis='x'); plt.tight_layout(); plt.show()

### Interpreting the ablations

**Depth (`n_layers`).** This is the most effective lever at a given compute
budget. Each extra block allows one more "reasoning pass": block 1 can identify
the previous token, block 2 can combine that with something else, and so on.
Note however that at only 200 iterations an 8-block model has not finished
converging: models should be compared at equal training budget, not merely at
equal iteration count.

**Number of heads (`n_heads`).** At fixed `n_embd`, increasing the number of
heads does **not** change the parameter count: the 128 dimensions are simply
split into 4 blocks of 32 instead of 1 block of 128. Each head can then
specialise on a different kind of relation (the previous token, the subject of
the verb, the matching opening bracket...). A single head degrades the results;
going from 4 to 8 brings little here, because head_dim drops to 16, which
becomes too small to represent a useful relation.

**Context (`block_size`).** More context means more available information, but
the cost of attention grows as **O(T^2)**: doubling the context quadruples the
cost of the attention matrix. This is THE structural limit of the Transformer,
and the reason every "efficient attention" variant exists (Longformer,
FlashAttention, Mamba...). On char-level text a 32-character context is clearly
insufficient: the model does not even see a full sentence.

**Dropout.** Over 200 iterations the model has no time to overfit, so
`dropout=0` is often best here. Over a long training run the opposite would
hold. A useful reminder: the optimal value of a regularisation hyperparameter
depends on the training duration.

**Width (`n_embd`).** Reducing it to 64 roughly quarters the block parameter
count and degrades results markedly. Width and depth are the two axes of
scaling, and the scaling laws show they must be increased together.

### Effect of temperature at generation time

The temperature $T$ divides the logits before the softmax:
$p_i = \mathrm{softmax}(z_i / T)$.

In [ ]:
meilleur = resultats["profondeur : 8 blocs"]["model"]
globals()["config"] = dict(base); globals()["config"]["n_layers"] = 8

for temp in (0.2, 0.5, 0.8, 1.0, 1.5):
    idx0 = torch.tensor([encode("\n")[:1] or [0]], dtype=torch.long, device=device)
    out = generate(meilleur, idx0, max_new_tokens=150, temperature=temp)
    print(f"\n=== temperature = {temp} ===")
    print(decode(out[0].tolist()))

- **$T \to 0$**: the distribution concentrates on the most likely token
  (equivalent to greedy decoding). Locally very clean text, but repetitive — it
  quickly falls into loops.
- **$T = 1$**: sampling follows the learned distribution as it is.
- **$T > 1$**: the distribution flattens and more risk is taken. More creative,
  but it ends up producing gibberish.

This is exactly the `temperature` parameter of LLM APIs.

# Partie B — Fine-tuning d’un GPT-2 pré-entraîné (GPU recommandé)

Cette partie montre un workflow réaliste de fine-tuning avec Hugging Face.

In [ ]:
# Installation
# !pip -q install -U transformers datasets accelerate

import torch
print("CUDA available:", torch.cuda.is_available())

## B1 — Charger GPT-2 + tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

gpt2 = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print("Loaded:", model_name)

# Inspectons le modèle:
print(gpt2)

## B2 — Préparer le dataset (GPT-2 tokenizer)

In [ ]:
from datasets import Dataset

# Utilisation du texte de Tiny Shakespeare chargé en Partie A
# On crée un Dataset HuggingFace à partir de la variable 'text'
ds = Dataset.from_dict({"text": [text]})

# Taille des blocs de tokens utilisés pour le fine-tuning
block_size_ft = 128

def tokenize_fn(examples):
    """Tokeniser le texte brut."""
    # On utilise le tokenizer GPT-2 défini en cell 36d002a5
    return tokenizer(examples["text"])

# Appliquer la tokenisation
tok = ds.map(tokenize_fn, batched=True, remove_columns=["text"])

def group_texts(examples):
    """Regrouper les tokens en blocs de longueur fixe."""
    # Concaténer tous les tokens du batch
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}

    # Calculer la longueur totale multiple de block_size_ft
    total_length = len(concatenated[list(examples.keys())[0]])
    total_length = (total_length // block_size_ft) * block_size_ft

    # Découper en blocs
    result = {}
    for k, t in concatenated.items():
        t = t[:total_length]
        result[k] = [t[i : i + block_size_ft] for i in range(0, total_length, block_size_ft)]

    # Les labels sont une copie des input_ids pour le LM causal
    result["labels"] = result["input_ids"].copy()
    return result

# Créer le dataset final pour l'entraînement
lm_ds = tok.map(group_texts, batched=True)

# Split train / validation (90% / 10%)
lm_ds = lm_ds.train_test_split(test_size=0.1)
train_ds = lm_ds["train"]
val_ds = lm_ds["test"]

print(f"Train size: {len(train_ds)} blocks, Val size: {len(val_ds)} blocks")

## B3 — Fine-tuning avec Trainer (GPU recommandé)

In [ ]:
# Dans cette partie, on utilise le Trainer de HuggingFace pour fine-tuner GPT-2
# sur notre petit dataset de langage causal. On compare une génération avant
# et après fine-tuning afin d'observer l'effet de l'entraînement sur le style
del model # Supprimons la référence au MiniGPT pour éviter les confusions

import torch
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 1. Définir le data collator (MLM=False pour causal language modeling)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 2. Définir les arguments d'entraînement
args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    max_steps=200,
    warmup_steps=20,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    eval_strategy="steps",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# 3. Créer le Trainer en utilisant 'gpt2'
trainer = Trainer(
    model=gpt2,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# 4. Génération avant le fine-tuning
prompt = "To be, or not to be:"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

print("=== Before fine-tune ===")
with torch.no_grad():
    gen_before = gpt2.generate(
        input_ids,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )
print(tokenizer.decode(gen_before[0], skip_special_tokens=True))

# 5. Fine-tuning
if torch.cuda.is_available():
    print("\nLancement de l'entraînement...")
    trainer.train()

    # 6. Génération après le fine-tuning
    print("\n=== After fine-tune ===")
    with torch.no_grad():
        gen_after = gpt2.generate(
            input_ids,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id
        )
    print(tokenizer.decode(gen_after[0], skip_special_tokens=True))
else:
    print("\nGPU absent : fine-tuning ignoré pour éviter un temps d'attente trop long.")

## B4 — Questions (fine-tuning)

1. Comparez qualitativement avant/après.
2. Pourquoi le fine-tuning converge-t-il plus vite que le from-scratch ?
3. (Option) augmenter `max_steps`.

### 1. Comparaison avant / apres

**Before**, GPT-2 produces perfectly grammatical modern English with no
connection to Shakespeare: it continues the prompt like a blog post or a
Wikipedia excerpt.

**After** a few hundred steps the *form* changes markedly: character names in
capitals followed by colons, short lines, archaic vocabulary (*thou*, *thee*,
*hath*). The content, however, often remains incoherent — 200 steps on a 1 MB
corpus are not enough to learn a plot.

This is the typical behaviour of short fine-tuning: it shifts **style** and
**format** long before substance. It is also why fine-tuning is so effective in
practice (adapting a model to a response format), and why it is not enough to
teach it new knowledge.

### 2. Why does fine-tuning converge faster?

Because it does not start from scratch. Our mini-GPT has to learn everything
from random weights: that characters form words, that words follow a syntax,
that dialogue has a structure. GPT-2 has already learned all of this on ~40 GB
of text (WebText).

Concretement :

- **the starting point is far better**: GPT-2's initial loss on Shakespeare is
  already lower than the *final* loss of our mini-GPT;
- **the representations are reusable**: the lower layers encode the general
  syntax of English, which does not change between Wikipedia and Shakespeare.
  Only the upper layers genuinely need adjusting;
- **the gradient is better conditioned**: training starts from an already
  "good" region of parameter space, hence a learning rate 10 to 100 times
  smaller (5e-5 against 3e-4) and convergence in hundreds of steps rather than
  tens of thousands.

This is the principle of **transfer learning**, and it is what makes deep
learning usable when only a few thousand examples are available. It already
appeared in this course in another form: ImageNet pre-training for vision.

### 3. Augmenter `max_steps`

Going from 200 to 2000 steps, the validation loss keeps falling and then
**rises again**: the corpus (1 MB) is tiny against GPT-2's 124 M parameters,
which start to memorise it. The usual remedies: early stopping on validation, a
lower learning rate, or training only a small subset of the weights (LoRA,
adapters) rather than the whole model.

### Recapitulatif : from scratch vs fine-tuning

| | Mini-GPT from scratch | Fine-tuning GPT-2 |
|---|---|---|
| Parameters | ~1 M | 124 M |
| Data required | everything must come from the corpus | corpus seen as a mere adjustment |
| Learning rate | 3e-4 | 5e-5 |
| Duration | thousands of steps | hundreds |
| Quality reached | plausible words | correct English + target style |
| Purpose | understanding the mechanism | what is actually done in practice |